In [1]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
import time
from cuml import KMeans
from cuml.cluster import KMeans

In [12]:
df_pandas = pd.read_csv('household_power_consumption_not_null.csv', parse_dates=[['Date', 'Time']],
                       date_format = {'Date': '%d/%m/%Y', 
                                      'Time': '%H:%M:%S'},
                       dayfirst = True)
df_pandas

/tmp/ipykernel_21780/4030125953.py:1: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df_pandas = pd.read_csv('household_power_consumption_not_null.csv', parse_dates=[['Date', 'Time']],


,Date_Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...,...
2049275,2010-11-26 20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0
2049276,2010-11-26 20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0
2049277,2010-11-26 21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0
2049278,2010-11-26 21:01:00,0.934,0.000,239.70,3.8,0.0,0.0,0.0


In [13]:
df_cudf_1 = cudf.DataFrame(df_pandas)
df_cudf_1

,Date_Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...,...
2049275,2010-11-26 20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0
2049276,2010-11-26 20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0
2049277,2010-11-26 21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0
2049278,2010-11-26 21:01:00,0.934,0.000,239.70,3.8,0.0,0.0,0.0


In [58]:
df_cudf_1 = df_cudf_1.astype(float)
df_cudf_1

,Date_Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,1.166290e+18,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,1.166290e+18,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,1.166290e+18,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,1.166290e+18,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,1.166290e+18,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...,...
2049275,1.290805e+18,0.946,0.000,240.43,4.0,0.0,0.0,0.0
2049276,1.290805e+18,0.944,0.000,240.00,4.0,0.0,0.0,0.0
2049277,1.290805e+18,0.938,0.000,239.82,3.8,0.0,0.0,0.0
2049278,1.290805e+18,0.934,0.000,239.70,3.8,0.0,0.0,0.0


In [65]:
class Clustering(object):

    def __init__(self, dataset):
        self.dataset = dataset.copy().reset_index(drop = True)

    def KMeans(self):
        global kmeans_global
        global labels_global
        global cluster_centers_global
        
        kmeans_float = KMeans(n_clusters=15, max_iter=100000, tol=0.0001, verbose=False, random_state=None, init='scalable-k-means++', 
                              n_init='auto', oversampling_factor=2.0, max_samples_per_batch=100000, output_type=None)
        kmeans = kmeans_float.fit(self.dataset)
        kmeans_global = kmeans

        labels = kmeans_float.labels_
        labels_global = labels

        cluster_centers = kmeans_float.cluster_centers_
        cluster_centers_global = cluster_centers
        
        
    def main(self):
        st = time.time()
        self.KMeans()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [66]:
clust = Clustering(df_cudf_1)

In [67]:
clust.main()

Execution time: 3317.1759824752808 seconds


In [68]:
kmeans_global

KMeans()

In [69]:
labels_global

0          11
1          11
2          11
3          11
4          11
           ..
2049275     8
2049276     8
2049277     8
2049278     8
2049279     8
Length: 2049280, dtype: int32

In [70]:
cluster_centers_global

,0,1,2,3,4,5,6,7
0,1.261156e+18,1.367656,0.126133,243.155580,5.691054,1.308762,1.283699,8.882520
1,1.203542e+18,1.292450,0.093613,240.834315,5.451165,1.267378,1.580095,6.561136
2,1.236492e+18,1.232090,0.105312,242.299688,5.160407,1.339310,1.235977,7.276488
3,1.277899e+18,0.820604,0.145721,240.824472,3.529415,0.836202,0.859623,5.727386
4,1.178817e+18,0.954871,0.126661,237.907945,4.128510,1.367543,1.493104,5.000632
5,1.220018e+18,0.743059,0.126588,240.107205,3.204819,0.774139,0.917014,4.721910
6,1.244723e+18,0.852648,0.140145,241.069523,3.656371,0.758030,0.945977,6.032949
7,1.269540e+18,1.149411,0.117955,241.657080,4.809603,0.986400,1.226696,8.218543
8,1.286703e+18,1.084114,0.126876,241.421367,4.585709,1.062251,1.121164,6.808177
9,1.211780e+18,1.004319,0.148550,240.065038,4.303070,1.297321,1.381929,6.469931
